# PD-L1(ST): WSI 직접 추론 → 계정별 Grade PCC · CCC

WSI 조직 영역 선택 → 겹치는 패치 추출 → PD-L1 모델 추론 및 패치 NMS → level-0 좌표 병합 → 전역 NMS → CPS/Grade → 전문의 Grade 비교.
백엔드의 조직 마스크, MPP 보정, 패치 읽기, NMS, CPS 함수를 재사용합니다. Philips는 data envelope를 사용합니다. 조직 영역은 종양 ROI와 동일하지 않습니다.

CSV에서는 식별정보(프로젝트·슬라이드·계정)와 **전문의 CPS Grade, CPS 판독시간, AI 판독시간**만 읽습니다. JSON 및 CSV에 저장된 AI 점수는 사용하지 않습니다. 슬라이드마다 새로 한 번 추론하여 계정 간 공유합니다.

**입력은 KNUCH PD-L1 비교 결과 ZIP 안의 `계정별` 폴더입니다.** 압축을 풀어 `<로그인 ID>/비교결과.csv`와 JSON을 그대로 `standalone_detection/계정별`에 배치하세요. JSON은 읽지 않습니다. 기본 폴더에는 가상 계정 데이터가 없습니다. 별도 `examples/knuch_zip_export/계정별`은 실제 KNUCH ZIP 생성 함수로 만든 가상 형식 테스트 자료입니다.

CSV Grade 문자열 `CPS < 1` → 0, `1 ≤ CPS < 10` → 1, `CPS ≥ 10` → 2로 변환합니다. 판독시간 단위는 초입니다. CSV 판독시간은 기록 그대로 유지하고 이번 WSI 처리시간은 `inference_seconds`로 따로 측정합니다(모델 로딩 제외).


In [ ]:
from pathlib import Path
import sys
import json
import torch
from html import escape
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = str
    display = print

def show_table(rows):
    if not rows:
        return
    columns = list(rows[0])
    header = ''.join('<th>'+escape(str(k))+'</th>' for k in columns)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(r.get(k, '')))+'</td>'
                                for k in columns)+'</tr>' for r in rows)
    display(HTML('<table><tr>'+header+'</tr>'+body+'</table>'))

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'backend/app/ai_pipelines').is_dir())
for directory in (ROOT / 'backend', ROOT / 'standalone_detection'):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from pdl1_wsi import (load_reader_csvs, project_slide_manifest,
                      WSIInference, compare_readers)
from pdl1_pcc import save_pcc_csv

ACCOUNT_FOLDER = ROOT / 'standalone_detection/계정별'
# 화면 제목 PD-L1(ST)의 실제 프로젝트 저장 경로입니다.
PROJECT_DIR = ROOT / 'backend/uploads/IHC(PD-L1)'
CHECKPOINT = ROOT / 'backend/model/PDL1_ST_CPS_detection.pt'
OUTPUT_DIR = ROOT / 'standalone_detection/results'
DEVICE = 'cuda:0'  # 사용할 GPU 번호
BATCH_SIZE = 8
MAX_PATCHES = None  # None: 전체 WSI. 2 등 지정 시 동작 점검만 하고 PCC/CCC 계산 차단.
# KNUCH ZIP의 원본 CSV 형식을 그대로 읽습니다. 별도 열 변환 불필요.
# 예제만 실행하려면 아래 경로로 ACCOUNT_FOLDER를 바꾸세요.
# ACCOUNT_FOLDER = ROOT / 'standalone_detection/examples/knuch_zip_export/계정별'


## 입력 및 슬라이드 연결 확인

같은 프로젝트·모델·계정·슬라이드의 반복 판독은 KNUCH ZIP 통계와 동일하게 측정 시각, 측정 ID 순으로 최신 하나를 선택합니다. 없는 슬라이드를 이름 유사도로 연결하지 않습니다. 다른 환경에서 생성한 ID라면 검증한 `슬라이드 ID: 실제 WSI 경로` 매핑을 `manifest`에 지정할 수 있습니다.


In [ ]:
records = load_reader_csvs(ACCOUNT_FOLDER)
manifest = project_slide_manifest(PROJECT_DIR)
slide_ids = sorted({r['slide'] for r in records})
missing = set(slide_ids) - set(manifest)
if missing:
    raise ValueError(f'프로젝트에서 찾지 못한 슬라이드 ID: {sorted(missing)}')
show_table(records)
show_table([{'slide_id': sid, 'wsi': str(manifest[sid])} for sid in slide_ids])


## 실제 WSI 추론

모델 구조·5개 클래스 순서를 검증합니다. 0.5 µm/px 기준 512 입력, 패치 overlap 10 µm, 경계 제외 3 µm, 패치/전역 NMS IoU 0.3, CPS 계산 confidence ≥ 0.1을 사용합니다.

CPS = min(100, (양성 종양 + 양성 면역) / 전체 생존 종양 × 100). 종양 분모가 0인 경우 비교에서 제외합니다. 일부 패치만 처리한 결과에는 WSI CPS/Grade를 부여하지 않습니다.


In [ ]:
engine = WSIInference(CHECKPOINT, device=DEVICE)
print(f'추론 장치: {engine.device}')
predictions = {}
for sid in slide_ids:
    predictions[sid] = engine.predict(manifest[sid], batch_size=BATCH_SIZE,
                                      max_patches=MAX_PATCHES)
show_table([{'slide_id': sid, **{k: v for k, v in result.items() if k != 'counts'}}
            for sid, result in predictions.items()])


## 계정별 및 과반 합의 Grade PCC · CCC

슬라이드별 유효 전문의 Grade의 엄격한 과반을 사용합니다(최소 2명). 과반이 없으면 제외합니다. CSV에는 Grade만 있으므로 **전문의 CPS 중앙값은 계산하지 않습니다**.

PCC는 Grade 코드의 Pearson 상관, CCC는 `2 cov(x,y) / (var(x)+var(y)+(mean(x)-mean(y))²)`입니다(분산·공분산 1/N). 누락된 값은 쌍 단위 제외하며, 2쌍 미만 또는 상수 입력 등 정의 불가 사유를 결과에 표시합니다.


In [ ]:
pairs, account_results, consensus_slides, consensus_results = compare_readers(records, predictions)
for title, rows in [('슬라이드별 비교 및 판독시간', pairs), ('계정별 PCC · CCC', account_results),
                    ('슬라이드별 Grade 과반 합의', consensus_slides), ('종합 PCC · CCC', consensus_results)]:
    print(title)
    show_table(rows)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for filename, rows in [('슬라이드별_비교.csv', pairs), ('계정별_PCC_CCC.csv', account_results),
                       ('슬라이드별_합의.csv', consensus_slides), ('종합_PCC_CCC.csv', consensus_results)]:
    if rows:
        print(save_pcc_csv(rows, OUTPUT_DIR / filename))
(OUTPUT_DIR / 'wsi_inference.json').write_text(
    json.dumps(predictions, ensure_ascii=False, indent=2), encoding='utf-8')
